## 1) Make sure all of your installs and imports are good.

In [ ]:
import pandas as pd
import os
import glob
import re
from pathlib import Path

## 2) Reading in all of the files
Make sure that your input files are properly in their folders. This code is for tecplot files, but could be modified for hd5

In [2]:
def extract_tec_files(working_directory):
    work_dir = Path(working_directory)
    
    if not work_dir.exists():
        print(f"Directory {working_directory} does not exist!")
        return {}
    
    # Find all .tec files with the pattern name-XXX.tec
    tec_files = glob.glob(str(work_dir / "*.tec"))
    
    # Dictionary to store grouped files
    file_groups = {}
    
    # Pattern to match files like somename-000.tec, somename-001.tec
    pattern = re.compile(r'^(.+)-(\d{3})\.tec$')
    
    for file_path in tec_files:
        filename = os.path.basename(file_path)
        match = pattern.match(filename)
        
        if match:
            base_name = match.group(1)
            sequence_num = int(match.group(2))
            
            if base_name not in file_groups:
                file_groups[base_name] = []
            
            file_groups[base_name].append((sequence_num, file_path))
    
    # Sort files by number for each base name
    for base_name in file_groups:
        file_groups[base_name].sort(key=lambda x: x[0])
        # Keep file paths
        file_groups[base_name] = [file_path for _, file_path in file_groups[base_name]]
    
    return file_groups

In [3]:
# Example usage:
working_directory = "/Users/sanskriti/Documents/GitHub/saltyBiomass/software_module/pflotran_visualization"  # Change this to your actual directory
tec_file_groups = extract_tec_files(working_directory)

# Display results
for base_name, files in tec_file_groups.items():
    print(f"\nBase name: {base_name}")
    print(f"Number of files: {len(files)}")
    for i, file_path in enumerate(files):
        print(f"  {i:03d}: {os.path.basename(file_path)}")


Base name: test30
Number of files: 6
  000: test30-000.tec
  001: test30-001.tec
  002: test30-002.tec
  003: test30-003.tec
  004: test30-004.tec
  005: test30-005.tec

Base name: test31
Number of files: 6
  000: test31-000.tec
  001: test31-001.tec
  002: test31-002.tec
  003: test31-003.tec
  004: test31-004.tec
  005: test31-005.tec

Base name: test29
Number of files: 6
  000: test29-000.tec
  001: test29-001.tec
  002: test29-002.tec
  003: test29-003.tec
  004: test29-004.tec
  005: test29-005.tec


## 3) We're going to extract water activity.

In [5]:
def extract_water_activity(tec_file_groups):
    water_activity_data = {}
    
    for base_name, files in tec_file_groups.items():

        first_file = files[0]
        # Read the tecplot file header to find variable names
        with open(first_file, 'r') as f:
            lines = f.readlines()
        
        # Find the VARIABLES line
        variables_line = None
        data_start_line = 0
        
        for i, line in enumerate(lines):
            if line.strip().startswith('VARIABLES'):
                variables_line = line
                break
            elif line.strip().startswith('ZONE'):
                data_start_line = i + 1
                break
        
        if variables_line:
            # Extract variable names from VARIABLES line

            var_matches = re.findall(r'"([^"]*)"', variables_line)
            column_names = var_matches
            
            # Skip to data section (after ZONE line)
            for i, line in enumerate(lines):
                if line.strip().startswith('ZONE'):
                    data_start_line = i + 1
                    break
            
            # Read data starting from data_start_line
            df = pd.read_csv(first_file, sep='\s+', skiprows=data_start_line, 
                            header=None, names=column_names)

        # Check if gamma_H2O column 
        if 'Gamma H2O' in df.columns:
            water_activity_data[base_name] = {
                'file': first_file,
                'Gamma H2O': df['Gamma H2O'].values,
                'coordinates': {
                    'x': df['X'].values if 'X' in df.columns else None,
                    'y': df['Y'].values if 'Y' in df.columns else None,
                    'z': df['Z'].values if 'Z' in df.columns else None
                }
            }
            print(f"✓ Extracted water activity from {base_name}: {len(df)} data points")
        else:
            print("???? something went wrong. check if you have Gamma H2O")  
    return water_activity_data

# Extract water activity from all 000 files
water_activity_results = extract_water_activity(tec_file_groups)

# Display summary
print(f"\n=== Water Activity Extraction Summary ===")
print(f"Successfully extracted from {len(water_activity_results)} file groups:")
for base_name, data in water_activity_results.items():
    gamma_values = data['Gamma H2O']
    print(f"  {base_name}: min={gamma_values.min():.4f}, max={gamma_values.max():.4f}, mean={gamma_values.mean():.4f}")

✓ Extracted water activity from test30: 64 data points
✓ Extracted water activity from test31: 1000 data points
✓ Extracted water activity from test29: 64 data points

=== Water Activity Extraction Summary ===
Successfully extracted from 3 file groups:
  test30: min=1.0000, max=1.0000, mean=1.0000
  test31: min=0.9939, max=0.9939, mean=0.9939
  test29: min=1.0000, max=1.0000, mean=1.0000


<>:36: SyntaxWarning: invalid escape sequence '\s'
<>:36: SyntaxWarning: invalid escape sequence '\s'
/var/folders/9z/tkcd7rfx1bxd05989nnkhc7r0000gn/T/ipykernel_3878/802829440.py:36: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(first_file, sep='\s+', skiprows=data_start_line,


## 4) We're going to get carbon and methane flux from the files for Day 1, 2, and 3.

## 5) We're going to plot it

## 6) on the todo list: 
* integrate for total flux